In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.tsa.statespace.sarimax import SARIMAX
from datetime import datetime
from itertools import product
from sqlalchemy import create_engine, text
import sqlalchemy
import matplotlib.pyplot as plt
import warnings

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

LOG_TRANSFORM_CODES = ['851762']
FORECAST_STEPS = 24
MIN_PERIODS = 60

db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

input_date = datetime.now().date()

print(f"=== SARIMA 예측 시스템 (기존 DB 호환 버전) ({input_date}) ===")

engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

print("기존 테이블 스키마 확인 중...")

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT CONSTRAINT_NAME
        FROM information_schema.TABLE_CONSTRAINTS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'us_trade_monthly_data_with_forecast'
          AND CONSTRAINT_TYPE = 'UNIQUE'
    """))

    existing_keys = [row[0] for row in result.fetchall()]
    print(f"월별 테이블 UNIQUE KEY: {existing_keys}")

    if 'uniq_monthly_tracking' not in existing_keys:
        try:
            for key in existing_keys:
                try:
                    conn.execute(text(f"""
                        ALTER TABLE us_trade_monthly_data_with_forecast
                        DROP INDEX {key}
                    """))
                except:
                    pass

            conn.execute(text("""
                ALTER TABLE us_trade_monthly_data_with_forecast
                ADD UNIQUE KEY uniq_monthly_tracking (hs_code_6d, date, forecast_flag, input_date)
            """))
            print("월별 테이블 UNIQUE KEY 업데이트 완료")
        except Exception as e:
            print(f"UNIQUE KEY 업데이트 실패: {e}")

    conn.commit()

print("무역 데이터 로딩 중...")
trade_df = fetch_table_data(db_info, "us_trade_data")
trade_df['hs_code_6d'] = trade_df['hs_code'].astype(str).str[:6]
trade_df['date'] = pd.to_datetime(trade_df['date'])

print(f"총 {len(trade_df):,}개 레코드 로드 완료")

valid_codes = trade_df['hs_code_6d'].unique().tolist()

def forecast_sarima(df, date_col='date', value_col='expDlr', steps=14, use_log=False):
    try:
        ts = df.groupby(date_col)[value_col].sum().asfreq('M')
        if ts.isnull().any() or len(ts.dropna()) < MIN_PERIODS:
            return pd.Series(dtype='float64')

        if use_log:
            ts = np.log(ts)

        p = d = q = [0, 1]
        P = D = Q = [0, 1]
        s = 12

        param_combinations = list(product(p, d, q))
        seasonal_combinations = list(product(P, D, Q))
        total_combinations = list(product(param_combinations, seasonal_combinations))

        best_aic = np.inf
        best_model = None

        for (order, seasonal) in total_combinations:
            seasonal_order = (*seasonal, s)
            try:
                model = SARIMAX(ts, order=order, seasonal_order=seasonal_order)
                result = model.fit(disp=False, maxiter=50)
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_model = result
            except Exception:
                continue

        if best_model is None:
            return pd.Series(dtype='float64')

        forecast = best_model.forecast(steps=steps)
        if use_log:
            forecast = np.exp(forecast)

        forecast.index = pd.date_range(
            start=ts.index[-1] + pd.offsets.MonthEnd(1),
            periods=steps,
            freq='M'
        )
        return forecast

    except Exception:
        return pd.Series(dtype='float64')

print(f"총 {len(valid_codes):,}개 HS Code에 대해 예측 시작...")

forecast_list = []
model_count = 0

for code in tqdm(valid_codes, desc="SARIMA 예측 진행"):
    sub_df = trade_df[trade_df['hs_code_6d'] == code].copy()
    use_log = code in LOG_TRANSFORM_CODES

    forecast = forecast_sarima(
        sub_df[['date', 'expDlr']],
        steps=FORECAST_STEPS,
        use_log=use_log
    )

    if not forecast.empty:
        temp_df = pd.DataFrame({
            'hs_code_6d': code,
            'date': forecast.index,
            'expDlr': forecast.values,
            'forecast_flag': 1,
            'input_date': input_date
        })
        forecast_list.append(temp_df)
        model_count += 1

print(f"예측 완료: {model_count}개")

historical_df = trade_df.groupby(['hs_code_6d', 'date'], as_index=False)['expDlr'].sum()
historical_df['forecast_flag'] = 0
historical_df['input_date'] = input_date

if forecast_list:
    forecast_combined = pd.concat(forecast_list, ignore_index=True)
    monthly_combined = pd.concat([historical_df, forecast_combined], ignore_index=True)
else:
    monthly_combined = historical_df.copy()

monthly_combined['quarter'] = monthly_combined['date'].dt.to_period('Q').astype(str)

monthly_combined = (monthly_combined
    .sort_values(['hs_code_6d', 'date', 'forecast_flag', 'input_date'])
    .drop_duplicates(subset=['hs_code_6d', 'date', 'forecast_flag', 'input_date'], keep='last'))

print(f"월별 데이터 준비 완료: {len(monthly_combined):,}개")

monthly_combined['quarter_period'] = monthly_combined['date'].dt.to_period('Q')
quarterly_grouped = (
    monthly_combined
    .groupby(['hs_code_6d', 'quarter_period', 'input_date'], as_index=False)
    .agg({
        'expDlr': 'sum',
        'forecast_flag': 'max'
    })
)

quarterly_grouped['quarter'] = quarterly_grouped['quarter_period'].astype(str)
quarterly_grouped['date'] = quarterly_grouped['quarter_period'].dt.to_timestamp() + pd.offsets.QuarterEnd(0)
quarterly_grouped = quarterly_grouped[['hs_code_6d', 'quarter', 'expDlr', 'date', 'input_date', 'forecast_flag']]

quarterly_grouped = (quarterly_grouped
    .sort_values(['hs_code_6d', 'quarter', 'forecast_flag', 'input_date'])
    .drop_duplicates(subset=['hs_code_6d', 'quarter', 'forecast_flag', 'input_date'], keep='last'))

print(f"분기 데이터 준비 완료: {len(quarterly_grouped):,}개")

print("월별 데이터 업로드 시작...")

try:
    temp_monthly = f"temp_monthly_{int(datetime.now().timestamp())}"

    monthly_combined.to_sql(
        name=temp_monthly,
        con=engine,
        if_exists='replace',
        index=False
    )

    with engine.connect() as conn:
        conn.execute(text(f"""
            INSERT INTO us_trade_monthly_data_with_forecast
            (hs_code_6d, date, expDlr, forecast_flag, input_date, quarter)
            SELECT hs_code_6d, date, expDlr, forecast_flag, input_date, quarter
            FROM {temp_monthly}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                quarter = VALUES(quarter);
        """))
        conn.execute(text(f"DROP TABLE {temp_monthly}"))
        conn.commit()

    print("월별 데이터 업로드 완료")
    monthly_success = True
except Exception as e:
    print(f"월별 업로드 오류: {e}")
    monthly_success = False

print("분기 데이터 업로드 시작...")

try:
    temp_quarterly = f"temp_quarterly_{int(datetime.now().timestamp())}"

    quarterly_grouped.to_sql(
        name=temp_quarterly,
        con=engine,
        if_exists='replace',
        index=False
    )

    with engine.connect() as conn:
        conn.execute(text(f"""
            INSERT INTO us_trade_quarter_data_with_forecast
            (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
            SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
            FROM {temp_quarterly}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                date = VALUES(date);
        """))
        conn.execute(text(f"DROP TABLE {temp_quarterly}"))
        conn.commit()

    print("분기 데이터 업로드 완료")
    quarterly_success = True
except Exception as e:
    print(f"분기 업로드 오류: {e}")
    quarterly_success = False

print("\n" + "="*60)
print("예측 시스템 실행 완료")
print("="*60)
print(f"예측 실행 날짜 (input_date): {input_date}")
print(f"예측 성공 HS Code: {model_count}개")
print(f"월별 레코드: {len(monthly_combined):,}개")
print(f"분기 레코드: {len(quarterly_grouped):,}개")

if monthly_success and quarterly_success:
    print("\n[완료] 모든 데이터 업로드 성공")
    print("\n예측 변화 추적 방법:")
    print("  - input_date를 예측 실행 날짜로 사용")
    print("  - 같은 (hs_code, date)에 대해 여러 input_date의 예측값 비교 가능")
else:
    print("\n[경고] 일부 데이터 업로드 실패")

print("\n프로그램 종료")


=== SARIMA 예측 시스템 (기존 DB 호환 버전) (2025-11-05) ===
기존 테이블 스키마 확인 중...
월별 테이블 UNIQUE KEY: ['uniq_monthly_tracking']
무역 데이터 로딩 중...
✅ 'us_trade_data' 테이블에서 68038건의 데이터를 가져왔습니다.
총 68,038개 레코드 로드 완료
총 478개 HS Code에 대해 예측 시작...


SARIMA 예측 진행: 100%|██████████| 478/478 [38:40<00:00,  4.86s/it] 


예측 완료: 447개
월별 데이터 준비 완료: 78,766개
분기 데이터 준비 완료: 26,578개
월별 데이터 업로드 시작...
월별 데이터 업로드 완료
분기 데이터 업로드 시작...
분기 데이터 업로드 완료

예측 시스템 실행 완료
예측 실행 날짜 (input_date): 2025-11-05
예측 성공 HS Code: 447개
월별 레코드: 78,766개
분기 레코드: 26,578개

[완료] 모든 데이터 업로드 성공

예측 변화 추적 방법:
  - input_date를 예측 실행 날짜로 사용
  - 같은 (hs_code, date)에 대해 여러 input_date의 예측값 비교 가능

프로그램 종료
